# Session 5 — Turbulence, Wakes, and Engineering Diagnostics

**Post-CFD Analysis with Python | Dr. Nuha Aljuneidi**

Downstream of any bluff body or obstruction, CFD reports a wake: reduced mean velocity and elevated turbulence. This session builds the standard diagnostics engineers use to characterize a wake and to compare turbulence-model predictions of it.

## Learning outcomes
- Compute turbulence intensity from a velocity field or fluctuation statistics.
- Extract and interpret a mean-velocity wake profile.
- Quantify wake deficit and wake half-width.
- Compare wake predictions from two synthetic turbulence models.

## Using your own Fluent or CSV data
This notebook uses synthetic data so you can run every cell immediately without a CFD license. When you are ready to use your own results, export a CSV from Fluent (or any solver) with coordinates, variable names, units, operating conditions, and a case identifier, then replace the synthetic-data cell below with:

```python
df = pd.read_csv("your_export.csv")
```

Map your solver's column names to the ones used in this notebook before continuing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(31)
print("Environment ready.")

## 1. Turbulence intensity

Turbulence intensity nondimensionalizes velocity fluctuation against the mean flow:

$$TI = \frac{\sqrt{\tfrac{1}{3}(u'^2+v'^2+w'^2)}}{U_{ref}}$$

Fluent typically reports either the fluctuating RMS directly or turbulent kinetic energy $k$, from which $TI = \sqrt{2k/3}/U_{ref}$.

In [ ]:
U_inf_mps = 12.0

y_m = np.linspace(-0.3, 0.3, 150)
# Turbulent kinetic energy elevated near the wake centerline (y=0), decaying with distance.
# Scaled so the near-wake peak lands in the physically typical 10-40% turbulence-intensity range,
# with a low freestream baseline away from the wake.
k_m2ps2 = 13.0 * np.exp(-(y_m ** 2) / 0.01) + 0.3 + rng.normal(0, 0.1, len(y_m)).clip(min=0)

TI = np.sqrt(2 * k_m2ps2 / 3) / U_inf_mps

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(TI * 100, y_m)
ax.set_xlabel("Turbulence intensity (%)")
ax.set_ylabel("y (m)")
ax.set_title("Turbulence intensity profile")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

if TI.max() < 0.05:
    ti_quality_msg = "unusually low for a near-wake peak, check k or U_ref"
elif TI.max() > 0.5:
    ti_quality_msg = "unusually high, check k or U_ref"
else:
    ti_quality_msg = "physically reasonable (wake TI typically 10-40%)"

print(f"Peak TI: {TI.max()*100:.1f}%  Freestream TI: {TI[np.argmax(np.abs(y_m)>0.28)]*100:.1f}% "
      f"— {ti_quality_msg}")

### Checkpoint 1 — Reference velocity matters
Recompute `TI` using a reference velocity of `U_inf_mps / 2` instead of `U_inf_mps`. How much does the reported peak turbulence intensity change? State, in one sentence, why the reference velocity used for $TI$ must always be reported alongside the number.

## 2. Mean-velocity wake profile

Directly behind a bluff body, mean streamwise velocity dips below freestream. Extract a lateral (`y`) profile at a fixed downstream distance and identify the deficit.

In [ ]:
x_over_D = 3.0  # downstream distance in body-diameters
D_m = 0.05

deficit_fraction = 0.55 * np.exp(-x_over_D / 6)  # deficit weakens with downstream distance
# Wake half-width targeted directly as a fraction of body diameter, growing slowly downstream --
# keeps the synthetic wake physically plausible (order of a diameter, not several diameters wide).
target_halfwidth_m = D_m * (0.3 + 0.05 * x_over_D)
sigma2 = target_halfwidth_m ** 2 / np.log(2)  # deficit reaches half its peak at target_halfwidth_m

velocity_wake_mps = U_inf_mps * (1 - deficit_fraction * np.exp(-(y_m ** 2) / sigma2))
velocity_wake_mps += rng.normal(0, 0.05, len(y_m))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(velocity_wake_mps, y_m)
ax.axvline(U_inf_mps, color="k", linestyle="--", linewidth=1, label="U_inf")
ax.set_xlabel("velocity_x (m/s)")
ax.set_ylabel("y (m)")
ax.set_title(f"Wake profile at x/D = {x_over_D:.1f}")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Wake deficit and wake half-width

**Velocity deficit** is how far below freestream the wake minimum is; **wake half-width** is the lateral distance from the centerline to where the deficit drops to half its peak value — a standard measure of wake spreading.

In [ ]:
def wake_metrics(y, velocity, U_ref):
    deficit_profile = U_ref - velocity
    centerline_idx = np.argmax(deficit_profile)
    peak_deficit_mps = deficit_profile[centerline_idx]
    half_level = peak_deficit_mps / 2

    # Find half-width on the positive-y side by locating where deficit crosses half_level
    y_pos = y[y >= y[centerline_idx]]
    deficit_pos = deficit_profile[y >= y[centerline_idx]]
    below_half = np.where(deficit_pos < half_level)[0]
    half_width_m = y_pos[below_half[0]] - y[centerline_idx] if len(below_half) else np.nan

    return peak_deficit_mps, half_width_m, centerline_idx

peak_deficit_mps, half_width_m, centerline_idx = wake_metrics(y_m, velocity_wake_mps, U_inf_mps)

print(f"Peak velocity deficit: {peak_deficit_mps:.3f} m/s "
      f"({100*peak_deficit_mps/U_inf_mps:.1f}% of U_inf) at y = {y_m[centerline_idx]:.3f} m")
print(f"Wake half-width: {half_width_m*1000:.1f} mm "
      f"({half_width_m/D_m:.2f} body diameters) at x/D = {x_over_D:.1f}")
print("Quality check: half-width should be a small fraction of a diameter this close behind the body — "
      f"{'consistent with near-wake expectations' if 0.1 < half_width_m/D_m < 1.0 else 'check profile extraction'}")

### Checkpoint 2 — Wake decay
Repeat the deficit-and-half-width calculation for `x_over_D = 1.0` and `x_over_D = 8.0` (regenerate `velocity_wake_mps` for each). Tabulate peak deficit and half-width vs. `x_over_D`. Does the deficit shrink and the half-width grow with downstream distance, as expected for a spreading, decaying wake?

In [ ]:
# TODO: loop over x_over_D in [1.0, 3.0, 8.0], regenerate the wake profile for each,
# compute wake_metrics, and print a small comparison table


## 4. Comparing turbulence-model predictions

Different turbulence models (e.g., k-epsilon vs. k-omega SST) predict measurably different wake shapes for the same case. Compare two synthetic "model" wake profiles the way you would compare two Fluent runs.

In [ ]:
# Model A: broader, shallower wake (typical of a more diffusive model)
velocity_modelA_mps = U_inf_mps * (1 - 0.40 * np.exp(-(y_m ** 2) / 0.03))
# Model B: narrower, deeper wake (typical of a less diffusive model)
velocity_modelB_mps = U_inf_mps * (1 - 0.58 * np.exp(-(y_m ** 2) / 0.012))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(velocity_modelA_mps, y_m, label="Model A (e.g., standard k-epsilon)")
ax.plot(velocity_modelB_mps, y_m, label="Model B (e.g., k-omega SST)")
ax.axvline(U_inf_mps, color="k", linestyle="--", linewidth=0.8)
ax.set_xlabel("velocity_x (m/s)")
ax.set_ylabel("y (m)")
ax.set_title(f"Turbulence-model comparison at x/D = {x_over_D:.1f}")
ax.legend()
plt.tight_layout()
plt.show()

deficit_A, halfwidth_A, _ = wake_metrics(y_m, velocity_modelA_mps, U_inf_mps)
deficit_B, halfwidth_B, _ = wake_metrics(y_m, velocity_modelB_mps, U_inf_mps)
print(f"Model A: peak deficit {deficit_A:.3f} m/s, half-width {halfwidth_A*1000:.1f} mm")
print(f"Model B: peak deficit {deficit_B:.3f} m/s, half-width {halfwidth_B*1000:.1f} mm")
print(f"Difference in peak deficit: {100*abs(deficit_A-deficit_B)/deficit_A:.0f}% — "
      "large enough that model choice is an engineering decision, not a formality.")

### Checkpoint 3 — Model sensitivity as a limitation
Write a two-sentence engineering note explaining to a project reviewer why reporting a wake deficit without stating which turbulence model produced it is incomplete, using the numbers from Model A vs. Model B above as evidence.

## Graduate/Advanced Extension
Estimate the momentum-deficit-based drag coefficient from a wake profile using the momentum-deficit method: $D' \approx \rho \int u(U_\infty - u)\,dy$ over the profile (per unit span). Apply it to Model A and Model B above and compare the resulting drag estimates to each other. State one assumption this method relies on that a full surface-pressure integration (Session 3) does not.

## Exit ticket
In three sentences: define wake half-width in your own words, state which of Model A or Model B you would trust more and why (or why you can't tell from this synthetic exercise alone), and name one wake-diagnostic you would want for your own project's flow.

**Next:** Session 6 automates everything so far across multiple cases and turns it into a reproducible report.